# Data Visualization with Matplotlib

## Setup Notebook and Kernel

The file is placed in the folder `day_73/` and the kernel used is the uv .venv inside the root folder `100-DAYS-OF-PYTHON/`
<div style="background:rgba(220,220,220,0.3); border-left:4px solid DodgerBlue; padding:10px; width:800px; border-radius:6px;">
<strong style="color:DodgerBlue; display:inline-flex; align-items:center; gap:8px;"><span style="display:inline-flex; align-items:center; justify-content:center; width:20px; height:20px; border-radius:50%; background:DodgerBlue; color:white; font-size:13px; font-weight:700; font-family:Arial, sans-serif; line-height:1;">i</span> Note</strong>
<p style="margin-bottom:5px;">
The choice to go with a local kernel is because Google Colab forces steps to work from its platform.<br>  
Pandas works mainly with CPU so for now no need to use GPUs and the code can be committed.
</p>
</div>

### Get the Data

Either use the provided .csv file or (optionally) get fresh (the freshest?) data from running an SQL query on StackExchange:

Follow this link to run the query from [StackExchange](https://data.stackexchange.com/stackoverflow/query/675441/popular-programming-languages-per-over-time-eversql-com) to get your own .csv file

```sql
select dateadd(month, datediff(month, 0, q.CreationDate), 0) m, TagName, count(*)
from PostTags pt
join Posts q on q.Id=pt.PostId
join Tags t on t.Id=pt.TagId
where TagName in ('java','c','c++','python','c#','javascript','assembly','php','perl','ruby','visual basic','swift','r','object-c','scratch','go','swift','delphi')
and q.CreationDate < dateadd(month, datediff(month, 0, getdate()), 0)
group by dateadd(month, datediff(month, 0, q.CreationDate), 0), TagName
order by dateadd(month, datediff(month, 0, q.CreationDate), 0)
```

### Import Pandas

In [142]:
import sys, pandas as pd
from IPython.display import Markdown, display
print(sys.version)
print(pd.__version__)

3.13.11 (main, Dec  5 2025, 16:06:33) [Clang 17.0.0 (clang-1700.4.4.1)]
3.0.1


### Import Data from CSV and Analyse

In [143]:
df = pd.read_csv("QueryResults.csv", names=['DATE', 'TAG', 'POSTS'], header=0)

Read the first rows

In [144]:
df.head()

,DATE,TAG,POSTS
0,2008-07-01 00:00:00,c#,3
1,2008-08-01 00:00:00,assembly,8
2,2008-08-01 00:00:00,c,82
3,2008-08-01 00:00:00,c#,503
4,2008-08-01 00:00:00,c++,164


Read the last rows

In [145]:
df.tail()

,DATE,TAG,POSTS
2924,2026-02-01 00:00:00,php,37
2925,2026-02-01 00:00:00,python,279
2926,2026-02-01 00:00:00,r,79
2927,2026-02-01 00:00:00,ruby,5
2928,2026-02-01 00:00:00,swift,69


Show the shape of the DataFrame

In [146]:
df.shape

(2929, 3)

List of columns

In [147]:
df.columns

Index(['DATE', 'TAG', 'POSTS'], dtype='str')

Missing values from columns

In [148]:
df.isna()

,DATE,TAG,POSTS
0,False,False,False
1,False,False,False
2,False,False,False
3,False,False,False
4,False,False,False
...,...,...,...
2924,False,False,False
2925,False,False,False
2926,False,False,False
2927,False,False,False


In [149]:
df.isna().any() #any column that has at leas one missing value

DATE     False
TAG      False
POSTS    False
dtype: bool

In [150]:
df.isna().any(axis=1) #any row that has at leas one missing value

0       False
1       False
2       False
3       False
4       False
        ...  
2924    False
2925    False
2926    False
2927    False
2928    False
Length: 2929, dtype: bool

In [151]:
mask = df.isna().any(axis=1) #any row that has at leas one missing value, this is a 1D boolean mask aligned with index
df[mask] # use the mask to identify rows with at least one column with empty cells

,DATE,TAG,POSTS


In [152]:
df.count()

DATE     2929
TAG      2929
POSTS    2929
dtype: int64

## Challenges

### Number of posts per language

In [153]:
posts_per_language = df.groupby('TAG')['POSTS'].sum().sort_values(ascending=False) # return a Series, sorted
top_language = posts_per_language.index[0]
top_value = posts_per_language.iloc[0]

print(posts_per_language)
print(top_language, top_value)

TAG
javascript    2524473
python        2206951
java          1916381
c#            1622854
php           1463291
c++            814563
r              510566
c              408141
swift          336188
ruby           229227
go              74466
perl            68308
delphi          52553
assembly        45155
Name: POSTS, dtype: int64
javascript 2524473


In [154]:
display(Markdown(f'### The most discussed language is **{top_language}** with **{top_value:,}** posts.'))

### The most discussed language is **javascript** with **2,524,473** posts.

In [155]:
df.groupby('TAG')[['POSTS']].sum().sort_values(by='POSTS', ascending=False) # return a DataFrame, sorted, becayse there are two square brackets [['POSTS']]

,POSTS
TAG,
javascript,2524473
python,2206951
java,1916381
c#,1622854
php,1463291
c++,814563
r,510566
c,408141
swift,336188


### Month of posts per language

In [156]:
valid_rows = df['POSTS'] > 0 # create a series where there is at least 1 post
filtered_df = df[valid_rows] # use the series as a mask

months_per_language = filtered_df.groupby('TAG')['DATE'].count().sort_values(ascending=False) #group by TAG and count the number of months with at least 1 post
top_language = months_per_language.index[0]
top_value = months_per_language.iloc[0]
print(months_per_language)
print(top_language, top_value)

TAG
c#            212
assembly      211
c             211
c++           211
delphi        211
java          211
javascript    211
perl          211
php           211
python        211
ruby          211
r             209
swift         202
go            196
Name: DATE, dtype: int64
c# 212


In [157]:
display(Markdown(f'### The most discussed language is **{top_language}** with **{top_value:,}** months.'))

### The most discussed language is **c#** with **212** months.